# Notebook 05: AlphaFold - Protein Structure Prediction

**Lecture 10: Neural Networks and Deep Learning**  
**Center for Neuroscience and Cell Biology, University of Coimbra**

---

## Introduction

AlphaFold represents one of the most significant breakthroughs in computational biology. In this notebook, we'll explore how to use AlphaFold through Google Colab and understand the deep learning architecture behind it.

**Learning Objectives:**
- Understand the AlphaFold architecture and its innovations
- Learn to use AlphaFold for protein structure prediction
- Interpret structure predictions and confidence scores
- Apply AlphaFold to neuroscience-relevant proteins

**Applications:**
- Drug discovery and design
- Understanding disease mechanisms
- Protein engineering
- Structure-function relationships

## 1. AlphaFold Overview

### The Protein Folding Problem

- **Challenge:** Predict 3D structure from amino acid sequence
- **Levinthal Paradox:** Astronomical number of possible conformations
- **Importance:** Structure determines function

### AlphaFold Evolution

**AlphaFold 1 (2018)**
- ResNet + attention mechanisms
- CASP13 winner (Z-score > 2.5 on hard targets)

**AlphaFold 2 (2020)**
- Revolutionary Evoformer architecture
- Invariant Point Attention (IPA)
- CASP14: 92.4 GDT score ("solved the problem")

**AlphaFold 3 (2024)**
- Diffusion model architecture
- Predicts protein-ligand, protein-nucleic acid complexes
- Broader biomolecular applications

### Key Innovations

1. **Evoformer Architecture:**
   - Alternating row/column attention on multiple sequence alignments (MSA)
   - Pair representation for residue-residue relationships

2. **Invariant Point Attention (IPA):**
   - 3D equivariant attention mechanism
   - Respects geometric structure

3. **Structure Module:**
   - Iterative refinement
   - Outputs 3D coordinates + confidence scores

## 2. Installation and Setup

**Note:** AlphaFold requires significant computational resources (GPU with 16+ GB RAM). This notebook is best run in Google Colab with GPU runtime.

For this demonstration, we'll use **ColabFold**, a simplified interface to AlphaFold that's optimized for Colab.

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Not running in Google Colab - some features may be limited")

# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    print(f"\nGPU detected: {gpu_info[0]}")
    print("✅ Ready for AlphaFold predictions")
else:
    print("\n⚠️ No GPU detected - predictions will be slow")
    print("In Colab: Runtime -> Change runtime type -> GPU")

## 3. Install ColabFold

ColabFold provides a streamlined interface to AlphaFold with faster MSA generation using MMseqs2.

In [ ]:
# Install ColabFold (uncomment to run)
if IN_COLAB:
    # !pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
    # !pip install -q jax==0.4.13 jaxlib==0.4.13+cuda11.cudnn86 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    print("ColabFold installed")
else:
    print("Installation instructions: https://github.com/sokrypton/ColabFold")

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

## 4. Example: Predict Structure of NMDA Receptor Subunit

Let's predict the structure of a neuroscience-relevant protein: the NMDA receptor GluN1 subunit ligand-binding domain.

In [ ]:
# Example protein sequence: NMDA receptor GluN1 subunit (partial)
# This is a truncated version for demonstration
sequence = """MSSWVNVTETVRKTPPRNYGANMSIGGFPKDLHIIPRVLWLLLFAFSLGWYFFAVVAE
QCQVLHHWKDSTLSTNFSIVAMLEDAFPGLLDDDIMIGINNPVFGDTMTTILVQYAEH
GPMECVFGKEMQQRGQVYRYLTKVPRVFDGTRNLFNYTVSKHFQHPRLLVLKSKQLK""".replace('\n', '')

print(f"Protein sequence (length {len(sequence)}):")
print(sequence[:60] + "..." if len(sequence) > 60 else sequence)
print(f"\nFull sequence length: {len(sequence)} amino acids")

## 5. Running AlphaFold Prediction

**Note:** This is a conceptual demonstration. In practice:
- Use [Google Colab's AlphaFold notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)
- Or access [AlphaFold Server](https://alphafoldserver.com/) for web-based predictions

The prediction pipeline involves:
1. **MSA Generation:** Search databases for homologous sequences
2. **Template Search:** Find similar known structures
3. **Structure Prediction:** Run AlphaFold neural network
4. **Confidence Estimation:** Calculate pLDDT and PAE scores

In [ ]:
# Pseudo-code for AlphaFold prediction
# (Actual implementation requires ColabFold setup)

prediction_example = """
# In ColabFold:
from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir

# Setup
queries = [("NMDA_GluN1", sequence)]
output_dir = Path("predictions")
output_dir.mkdir(exist_ok=True)

# Run prediction
run(
    queries=queries,
    result_dir=output_dir,
    num_models=5,  # Generate 5 models
    num_recycles=3,  # Number of recycling iterations
    model_type="auto",
    use_templates=True,
    use_amber=True,  # Energy minimization
)
"""

print("AlphaFold Prediction Pipeline:")
print(prediction_example)
print("\n⚠️ Run this in Google Colab with GPU for actual predictions")
print("   Link: https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb")

## 6. Understanding AlphaFold Outputs

AlphaFold produces several important outputs:

In [ ]:
# Simulated AlphaFold output for demonstration
# In reality, these come from actual predictions

# 1. pLDDT (predicted Local Distance Difference Test)
# Per-residue confidence score (0-100)
# >90: Very high confidence
# 70-90: Confident
# 50-70: Low confidence
# <50: Very low confidence

# Simulated pLDDT scores
np.random.seed(42)
plddt_scores = np.random.uniform(60, 95, len(sequence))
# Add some low-confidence regions (loops)
plddt_scores[50:60] = np.random.uniform(40, 60, 10)
plddt_scores[120:130] = np.random.uniform(45, 65, 10)

# Visualize pLDDT
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# pLDDT line plot
axes[0].plot(plddt_scores, linewidth=2, color='#1f77b4')
axes[0].axhline(y=90, color='green', linestyle='--', label='Very high confidence', alpha=0.7)
axes[0].axhline(y=70, color='orange', linestyle='--', label='Confident', alpha=0.7)
axes[0].axhline(y=50, color='red', linestyle='--', label='Low confidence', alpha=0.7)
axes[0].fill_between(range(len(plddt_scores)), 0, plddt_scores, alpha=0.3)
axes[0].set_xlabel('Residue Position', fontsize=12)
axes[0].set_ylabel('pLDDT Score', fontsize=12)
axes[0].set_title('Per-Residue Confidence (pLDDT)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# pLDDT distribution
axes[1].hist(plddt_scores, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].axvline(x=np.mean(plddt_scores), color='red', linestyle='--', 
                label=f'Mean: {np.mean(plddt_scores):.1f}', linewidth=2)
axes[1].set_xlabel('pLDDT Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('pLDDT Score Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nConfidence Summary:")
print(f"Mean pLDDT: {np.mean(plddt_scores):.2f}")
print(f"Very high confidence (>90): {(plddt_scores > 90).sum()} / {len(plddt_scores)} residues")
print(f"Confident (70-90): {((plddt_scores >= 70) & (plddt_scores <= 90)).sum()} / {len(plddt_scores)} residues")
print(f"Low confidence (<70): {(plddt_scores < 70).sum()} / {len(plddt_scores)} residues")

## 7. Predicted Aligned Error (PAE)

PAE measures confidence in relative positions between residues. Lower values indicate higher confidence.

In [ ]:
# Simulated PAE matrix
n_residues = len(sequence)
pae_matrix = np.random.uniform(0, 15, (n_residues, n_residues))

# Make diagonal more confident
for i in range(n_residues):
    for j in range(n_residues):
        distance = abs(i - j)
        if distance < 10:
            pae_matrix[i, j] = np.random.uniform(0, 3)
        elif distance < 30:
            pae_matrix[i, j] = np.random.uniform(3, 8)

# Visualize PAE
plt.figure(figsize=(10, 8))
plt.imshow(pae_matrix, cmap='viridis_r', vmin=0, vmax=30)
plt.colorbar(label='Expected Position Error (Å)', shrink=0.8)
plt.xlabel('Residue Position', fontsize=12)
plt.ylabel('Residue Position', fontsize=12)
plt.title('Predicted Aligned Error (PAE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPAE Interpretation:")
print("- Dark (low values): High confidence in relative position")
print("- Light (high values): Low confidence in relative position")
print("- Strong diagonal: Local structure is confident")
print("- Off-diagonal blocks: Confident domain-domain interactions")

## 8. Visualizing Protein Structures

AlphaFold outputs PDB files which can be visualized using various tools:

In [ ]:
# Visualization options
visualization_tools = {
    "py3Dmol": "Interactive 3D visualization in Jupyter/Colab",
    "PyMOL": "Professional molecular visualization software",
    "ChimeraX": "Modern molecular visualization from UCSF",
    "Mol*": "Web-based viewer (https://molstar.org/)",
    "Protein Imager": "High-quality publication figures"
}

print("Protein Structure Visualization Tools:\n")
for tool, description in visualization_tools.items():
    print(f"• {tool}: {description}")

print("\n" + "="*60)
print("Example: Using py3Dmol in Colab")
print("="*60)

py3dmol_example = """
# Install and import
!pip install py3Dmol
import py3Dmol

# Load and visualize structure
view = py3Dmol.view(width=800, height=600)
view.addModel(open('prediction.pdb').read(), 'pdb')

# Color by confidence (pLDDT)
view.setStyle({'cartoon': {
    'colorscheme': {
        'prop': 'b',  # B-factor column (contains pLDDT)
        'gradient': 'roygb',
        'min': 50,
        'max': 100
    }
}})

view.zoomTo()
view.show()
"""

print(py3dmol_example)

## 9. AlphaFold Database

The AlphaFold Database contains pre-computed predictions for over 200 million proteins.

In [ ]:
# Access AlphaFold Database
print("AlphaFold Database Resources:\n")

resources = {
    "AlphaFold Database": "https://alphafold.ebi.ac.uk/",
    "Download structures": "https://alphafold.ebi.ac.uk/download",
    "API access": "https://alphafold.ebi.ac.uk/api-docs",
    "AlphaFold Server": "https://alphafoldserver.com/",
}

for name, url in resources.items():
    print(f"• {name}:")
    print(f"  {url}\n")

# Example: Fetch structure from database
fetch_example = """
# Fetch pre-computed structure by UniProt ID
import requests

uniprot_id = "Q05586"  # Example: NMDA receptor GluN1
url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"

response = requests.get(url)
with open(f"{uniprot_id}_alphafold.pdb", 'w') as f:
    f.write(response.text)

print(f"Downloaded structure for {uniprot_id}")
"""

print("Example: Fetching Pre-computed Structure")
print("="*60)
print(fetch_example)

## 10. Applications in Neuroscience

AlphaFold has revolutionized structural neuroscience:

In [ ]:
neuroscience_applications = {
    "Ion Channels": {
        "examples": ["NMDA receptors", "AMPA receptors", "Voltage-gated channels"],
        "impact": "Understanding gating mechanisms, drug binding sites"
    },
    "GPCRs": {
        "examples": ["Dopamine receptors", "Serotonin receptors", "Opioid receptors"],
        "impact": "Drug discovery for psychiatric and neurological disorders"
    },
    "Synaptic Proteins": {
        "examples": ["PSD-95", "Synapsin", "Neurexin-Neuroligin"],
        "impact": "Understanding synaptic plasticity and autism-related mutations"
    },
    "Neurodegenerative Proteins": {
        "examples": ["Tau", "Amyloid-beta", "Alpha-synuclein"],
        "impact": "Designing therapeutics for Alzheimer's and Parkinson's"
    },
    "Transcription Factors": {
        "examples": ["CREB", "NPAS4", "c-Fos"],
        "impact": "Understanding activity-dependent gene regulation"
    }
}

print("AlphaFold Applications in Neuroscience\n")
print("="*70)

for category, info in neuroscience_applications.items():
    print(f"\n{category}:")
    print(f"  Examples: {', '.join(info['examples'])}")
    print(f"  Impact: {info['impact']}")

print("\n" + "="*70)
print("\n📊 Impact Metrics (as of 2024):")
print("  • 200M+ predicted structures")
print("  • 10,000+ citations")
print("  • 1M+ researchers using AlphaFold")
print("  • Accelerated drug discovery by 5-10x")

## 11. Best Practices and Limitations

In [ ]:
print("✅ When AlphaFold Works Best:\n")
best_practices = [
    "Single-chain proteins (<2700 amino acids)",
    "Globular, structured domains",
    "Proteins with evolutionary information (homologs)",
    "Static structures (not dynamic conformations)",
    "Overall fold prediction (backbone accuracy)"
]
for i, practice in enumerate(best_practices, 1):
    print(f"  {i}. {practice}")

print("\n⚠️ AlphaFold Limitations:\n")
limitations = [
    "Intrinsically disordered regions (low pLDDT)",
    "Conformational changes and dynamics",
    "Ligand binding effects (use AlphaFold 3)",
    "Membrane protein orientations",
    "Post-translational modifications",
    "Protein-protein interfaces (sometimes inaccurate)",
    "Side chain orientations (less accurate than backbone)"
]
for i, limitation in enumerate(limitations, 1):
    print(f"  {i}. {limitation}")

print("\n💡 Recommendations:\n")
recommendations = [
    "Always check pLDDT and PAE scores",
    "Validate predictions experimentally when possible",
    "Use AlphaFold 3 for protein complexes",
    "Combine with molecular dynamics for dynamics",
    "Compare multiple models (ensemble)",
    "Use database predictions when available (faster)"
]
for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

## 12. Hands-On Exercise

**Task:** Use the AlphaFold Colab notebook to predict the structure of a protein of your choice.

**Steps:**
1. Go to the [ColabFold notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)
2. Select a protein from your research or choose from examples:
   - **NMDA Receptor GluN1:** UniProt Q05586
   - **PSD-95:** UniProt P78352
   - **Tau protein:** UniProt P10636
3. Input the sequence or UniProt ID
4. Run the prediction (takes 5-20 minutes)
5. Analyze the outputs:
   - pLDDT scores
   - PAE matrix
   - 3D structure visualization

**Questions to Consider:**
- What is the overall confidence (mean pLDDT)?
- Are there low-confidence regions? What might they represent?
- How does the PAE matrix look? Are there distinct domains?
- Does the structure match known experimental structures (if available)?
- What biological insights can you gain from the structure?

## Summary

In this notebook, we explored:

1. ✅ AlphaFold's revolutionary architecture and innovations
2. ✅ How to use ColabFold for structure prediction
3. ✅ Interpreting confidence scores (pLDDT, PAE)
4. ✅ Accessing the AlphaFold Database
5. ✅ Applications in neuroscience research
6. ✅ Best practices and limitations

**Key Takeaways:**
- AlphaFold has solved the 50-year protein folding problem
- Confidence scores (pLDDT, PAE) are critical for interpretation
- Pre-computed predictions available for 200M+ proteins
- AlphaFold 3 extends to protein complexes and small molecules
- Experimental validation remains important

**Resources:**
- [AlphaFold Database](https://alphafold.ebi.ac.uk/)
- [ColabFold GitHub](https://github.com/sokrypton/ColabFold)
- [AlphaFold Paper (Nature 2021)](https://www.nature.com/articles/s41586-021-03819-2)
- [AlphaFold 3 Paper (Nature 2024)](https://www.nature.com/articles/s41586-024-07487-w)

**Next Steps:**
- Try predicting structures for your research proteins
- Explore AlphaFold 3 for protein-ligand complexes
- Combine with molecular dynamics simulations
- Use for rational drug design